# ESMT Ranking Intelligence Radar

Reproducible management analysis for the prototype seed as of **18 August 2026**. The notebook separates publisher-supported observations from ESMT interpretation. It does not predict a future rank.

## 1. Analysis contract

- Inputs: curated signal records plus small derived tables from the supplied official QS result exports.
- Output: priority diagnostics, component movement, and selected German-peer comparisons.
- Limitation: published scores describe outcomes; they do not establish causal effects of individual submitted fields or surveys.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data').exists(), 'Open the notebook from the project root or notebooks directory.'
plt.style.use('seaborn-v0_8-whitegrid')
BLUE, AMBER, GREY = '#165DFF', '#C47B13', '#667085'

In [ ]:
with (ROOT / 'data/processed/signals_enriched.json').open(encoding='utf-8') as handle:
    signals = pd.DataFrame(json.load(handle))
trends = pd.read_csv(ROOT / 'data/raw/qs_esmt_trends.csv')
peers = pd.read_csv(ROOT / 'data/raw/qs_german_peer_ranks.csv')
print(f'{len(signals)} signals | {len(trends)} ESMT edition rows | {len(peers)} peer rows')

## 2. Priority queue

The score is an auditable management rule. It is useful for sorting, but the score does not prove that one signal is objectively more important than another.

In [ ]:
priority = signals[['priority_score', 'priority_band', 'title', 'signal_type', 'status', 'proposed_owner']].copy()
priority.sort_values(['priority_score', 'title'], ascending=[False, True]).head(12)

In [ ]:
band_order = ['Act now', 'Plan', 'Monitor', 'Archive']
counts = signals['priority_band'].value_counts().reindex(band_order, fill_value=0)
ax = counts.plot(kind='barh', color=[BLUE, AMBER, GREY, '#B8C0CC'], figsize=(8, 3.5))
ax.set(title='Prototype priority distribution', xlabel='Signals', ylabel='')
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.tight_layout();

## 3. QS Global MBA: rank, score, and components

**Verified from the supplied QS exports:** ESMT moves 78 → 84 between the 2025 and 2026 editions, while its overall score moves 60.4 → 59.4. Entrepreneurship & Alumni Outcomes falls 5.4 points and Employability falls 2.1; the other displayed components improve.

**Interpretation:** the component declines define an investigation queue, not a causal explanation.

In [ ]:
mba = trends[trends['ranking'] == 'QS Global MBA'].set_index('edition')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mba['rank'].plot(marker='o', color=BLUE, ax=axes[0], title='ESMT QS Global MBA rank')
axes[0].invert_yaxis(); axes[0].set_ylabel('Rank (lower is better)')
mba['overall_score'].plot(marker='o', color=AMBER, ax=axes[1], title='ESMT QS Global MBA overall score')
axes[1].set_ylabel('Score')
plt.tight_layout();

In [ ]:
mba_components = ['employability', 'entrepreneurship_alumni_outcomes', 'roi', 'thought_leadership', 'diversity']
mba_delta = (mba.loc[2026, mba_components] - mba.loc[2025, mba_components]).sort_values()
colors = ['#BA3A3A' if value < 0 else BLUE for value in mba_delta]
ax = mba_delta.plot(kind='barh', color=colors, figsize=(9, 4), title='2026 minus 2025 component score')
ax.axvline(0, color='#101723', linewidth=0.8); ax.set_xlabel('Points'); ax.set_ylabel('')
plt.tight_layout();
mba_delta.to_frame('change')

In [ ]:
mba_peers = peers[peers['ranking'] == 'QS Global MBA'].pivot(index='school', columns='edition', values='rank')
mba_peers['rank_change_2026_vs_2025'] = mba_peers[2026] - mba_peers[2025]
mba_peers.sort_values(2026)

## 4. QS Management: why rank alone is misleading

**Verified from the supplied QS exports:** ESMT moves 75 → 100 while the overall score moves only 52.0 → 51.1. Value for Money falls 4.6 points and Diversity falls 3.1; Thought Leadership and Employability improve.

**Interpretation:** score density, entrants, and competitor movement must be examined before management describes the 25-place movement as a comparable decline in underlying performance.

In [ ]:
mim = trends[trends['ranking'] == 'QS Management'].set_index('edition')
mim_components = ['employability', 'alumni_outcomes', 'value_for_money', 'thought_leadership', 'diversity']
mim_delta = (mim.loc[2026, mim_components] - mim.loc[2025, mim_components]).sort_values()
mim_peers = peers[peers['ranking'] == 'QS Management'].pivot(index='school', columns='edition', values='rank')
mim_peers['rank_change_2026_vs_2025'] = mim_peers[2026] - mim_peers[2025]
display(mim_delta.to_frame('component_change'))
display(mim_peers.sort_values(2026))

## 5. Communication QA

The seed contains two verified year-label inconsistencies across ESMT and FT pages. These are not cosmetic only: duplicated claims without a controlled record create avoidable credibility and update risk.

In [ ]:
qa = signals[signals['signal_type'] == 'communication_mismatch'][
    ['title', 'factual_summary', 'recommended_action', 'proposed_owner', 'proposed_due_date']
]
qa

## 6. Management conclusion

The immediate queue is not ‘produce more news’. It is: validate near-term publication events, prepare upcoming QS collection windows, complete controlled reviews of FT50/AACSB/EQUIS changes, investigate QS component movements, and correct two public claim mismatches.

Before production, connect authorised private sources, create immutable snapshots, agree owners and escalation rules, and version every scoring or classification override.